# Six-terminal integer-QH transport: barrier vs barrierless comparison

Workflow: **(i)** build two otherwise identical six-terminal Hall bars using the **same disorder realization**, one with the $\nu$-dependent contact barriers and one without barriers, **(ii)** compute the common barrier-free bulk DOS at fixed $B$, **(iii)** obtain $\nu(E_F)$ from the finite-$T$ Fermi occupation, **(iv)** run Landauer–Büttiker transport for both systems with the same $E_F$ grid and temperature, and **(v)** compare $R_{xx}(\nu)$ and $R_{xy}(\nu)$.

Lead order is fixed: `0=source`, `1=drain`, `2=bottom-left`, `3=top-left`, `4=bottom-right`, `5=top-right`. Resistances are returned in units of $h/e^2$.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Folder containing qh6t.py
MODULE_DIR = Path.cwd()
if not (MODULE_DIR / "qh6t.py").exists():
    MODULE_DIR = Path("/mnt/data/qh6t_transport")
sys.path.insert(0, str(MODULE_DIR))

import qh6t
from qh6t import Geometry, Disorder, Barriers, Model


## 1. Parameters
All geometry is in nm and all energies are in meV. `a_nm` is the Kwant discretization length, not the GaAs crystal lattice constant.

In [ ]:
# ---------------- geometry ----------------
g = Geometry(
    Lx_nm=1200.0,
    Ly_nm=500.0,
    arm_width_nm=200.0,
    arm_length_nm=400.0,
    arm_centers_nm=(-250.0, 250.0),
    a_nm=10.0,
)

# ---------------- physics ----------------
B_T = 1.
T_K = 0.1
m_eff = 0.067

dis = Disorder(
    amplitude_meV=0.008,
    correlation_nm=20.0,
    seed=3,
)

# A order: source, drain, bottom-left, top-left, bottom-right, top-right
# A>0 activates below each integer N: [N(1-A), N].
# A<0 activates on the opposite side: [N, N(1-A)].  A=0 => always off.
bar = Barriers(
    A=(-0.2, -0.2, -0., -0., -0., -0.),
    width_nm=150.0,
    smooth_nm=50.0,
    height_cyclotron=1.,   # maximum = 1 * hbar*omega_c
)

model = Model(geometry=g, disorder=dis, barriers=bar, B_T=B_T, m_eff=m_eff)

# Identical reference model, but with all six contact barriers switched off.
bar0 = Barriers(
    A=(0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
    width_nm=bar.width_nm,
    smooth_nm=bar.smooth_nm,
    height_cyclotron=bar.height_cyclotron,
)
model0 = Model(geometry=g, disorder=dis, barriers=bar0, B_T=B_T, m_eff=m_eff)

hwc = qh6t.cyclotron_meV(B_T, m_eff)
print(f"hbar omega_c = {hwc:.4f} meV")
print(f"t = {qh6t.hopping_meV(g.a_nm, m_eff):.3f} meV")
print(f"flux/plaquette = {B_T*g.a_nm**2/qh6t.PHI0_T_NM2:.4f}")


## 2. Build the two Hall bars once
The disorder realization is generated once and explicitly reused in both systems. `fsyst` contains the $\nu$-dependent contact barriers, while `fsyst0` is the otherwise identical barrierless reference. The leads remain ideal in both cases.

In [ ]:
# Generate one disorder realization and reuse it exactly in both devices.
disorder = qh6t.make_disorder(model)

builder, _ = qh6t.build_system(model, disorder=disorder)
fsyst, magnetic_params = qh6t.finalize_with_B(builder, B_T, nleads=6)

builder0, _ = qh6t.build_system(model0, disorder=disorder)
fsyst0, magnetic_params0 = qh6t.finalize_with_B(builder0, B_T, nleads=6)

import kwant
kwant.plot(fsyst, site_size=0.18, lead_site_size=0.18);


## 3. Barrier-free DOS and the map $E_F\mapsto\nu$
The DOS calibration uses the central rectangular bulk region with the same disorder and magnetic field, but **without contact barriers**. The filling is $\nu=N_e/N_\Phi$, with $N_\Phi=AB/\Phi_0$.

In [ ]:
dos_E, dos_rho, n_sites_dos = qh6t.compute_dos(
    model, disorder,
    energy_resolution_meV=hwc/2000,
    num_vectors=None,
)

# Choose the transport window. Start broad; inspect nu(E), then tighten if useful.
E_min = 1.5 * hwc
E_max = 3.5 * hwc
N_E = 500
energies_uniform = np.linspace(E_min, E_max, N_E)

# Geometric area represented by the bulk DOS sample.
# Using N_sites*a^2 is consistent with the discrete KPM normalization.
area_dos_nm2 = n_sites_dos * g.a_nm**2
nu_uniform = qh6t.filling_from_dos(energies_uniform, dos_E, dos_rho, B_T, area_dos_nm2, T_K)



In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(dos_E/hwc, dos_rho)
ax[0].set(xlabel=r"$E/(\hbar\omega_c)$", ylabel="DOS [states/meV]")
ax[0].grid(alpha=.25)
ax[0].set_xlim(0,10)
ax[0].set_yscale('log')
ax[0].set_ylim(1e2,1e5)
ax[1].plot(energies_uniform/hwc, nu_uniform, marker='o')
ax[1].set(xlabel=r"$E_F/(\hbar\omega_c)$", ylabel=r"$\nu(E_F)$")
ax[1].grid(alpha=.25)
fig.tight_layout()

In [ ]:
def adaptive_energy_grid(
    energies,
    fillings,
    n_points,
    baseline=0.10,
    power=1.0,
):
    """
    Build a nonuniform energy grid concentrated where nu(E) changes.

    Parameters
    ----------
    energies : array
        Original uniformly sampled energy grid [meV].
    fillings : array
        nu(E) evaluated on `energies`.
    n_points : int
        Number of points in the new grid.
    baseline : float
        Fraction of uniform sampling retained in flat regions.
        0 -> all points follow dnu/dE
        1 -> completely uniform grid.
    power : float
        Controls concentration near steps.
        power > 1 concentrates more strongly.

    Returns
    -------
    E_new : ndarray
        Adaptive energy grid [meV].
    """
    E = np.asarray(energies, dtype=float)
    nu = np.asarray(fillings, dtype=float)

    # Where filling changes rapidly
    dnu_dE = np.abs(np.gradient(nu, E))

    # Normalize so the parameter baseline is easy to interpret
    if dnu_dE.max() > 0:
        weight_step = (dnu_dE / dnu_dE.max())**power
    else:
        weight_step = np.zeros_like(E)

    # Mix adaptive and uniform sampling
    weight = baseline + (1.0 - baseline) * weight_step

    # Cumulative sampling density
    cumulative = np.zeros_like(E)
    cumulative[1:] = np.cumsum(
        0.5 * (weight[1:] + weight[:-1]) * np.diff(E)
    )
    cumulative /= cumulative[-1]

    # Uniform points in cumulative coordinate -> nonuniform points in E
    q = np.linspace(0.0, 1.0, n_points)

    return np.interp(q, cumulative, E)

energies = adaptive_energy_grid(
    energies_uniform,
    nu_uniform,
    n_points=100,
    baseline=0.01,
    power=1.0,
)

nu = np.interp(
    energies,
    energies_uniform,
    nu_uniform,
)

energies = energies_uniform
nu = nu_uniform

plt.plot(energies, nu, marker='o')


## 4. Inspect the six $\nu$-dependent barrier switches
This is a cheap sanity check before the expensive transport sweep.

In [ ]:
nu_test = np.linspace(max(0, nu.min()), nu.max(), 1500)
names = ["source", "drain", "bottom-left", "top-left", "bottom-right", "top-right"]

fig, ax = plt.subplots(figsize=(8, 3.5))
for j, (name, A) in enumerate(zip(names, bar.A)):
    sw = np.array([qh6t.barrier_is_on(v, A) for v in nu_test], float)
    ax.plot(nu_test, sw + 1.15*j, label=f"{name}: A={A:g}")
ax.set(xlabel=r"$\nu$", ylabel="barrier switch + offset")
ax.grid(alpha=.25)
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()


## 5. Transport sweeps
The two calculations use exactly the same energy grid, filling calibration, magnetic field, temperature, geometry, disorder realization, and lead configuration. The only difference is the presence or absence of the six contact barriers.

In [ ]:
print("\n=== With barriers ===")
results = qh6t.transport_sweep(
    fsyst, magnetic_params,
    energies_meV=energies,
    fillings=nu,
    T_K=T_K,
    show_progress=True,
)

print("\n=== Without barriers ===")
results0 = qh6t.transport_sweep(
    fsyst0, magnetic_params0,
    energies_meV=energies,
    fillings=nu,
    T_K=T_K,
    show_progress=True,
)

# Optional save: prefix the barrierless arrays to keep both datasets in one file.
save_data = {**results, **{f"barrierless_{k}": v for k, v in results0.items()}}
np.savez_compressed(
    "six_terminal_QH_transport_barrier_comparison.npz",
    B_T=B_T, A=np.asarray(bar.A),
    **save_data,
)


## 6. Main result: barrier vs barrierless $R_{xx}(\nu)$ and $R_{xy}(\nu)$
The solid curves contain the $\nu$-dependent terminal barriers; the dashed curves are the otherwise identical barrierless reference.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7.5, 7), sharex=True)

ax[0].plot(results["nu"], results["Rxy"], lw=1.8, marker='o', label=r"$R_{xy}$ with barriers")
ax[0].plot(results0["nu"], results0["Rxy"], lw=1.8, ls="--", label=r"$R_{xy}$ without barriers")
ax[0].set_ylabel(r"$R_{xy}\ [h/e^2]$")
ax[0].grid(alpha=.25); ax[0].legend()

ax[1].plot(results["nu"], abs(results["Rxx"]), lw=1.8, label=r"$R_{xx}$ with barriers")
ax[1].plot(results0["nu"], abs(results0["Rxx"]), lw=1.8, ls="--", label=r"$R_{xx}$ without barriers")
ax[1].set(xlabel=r"filling $\nu$", ylabel=r"$R_{xx}\ [h/e^2]$")
ax[1].grid(alpha=.25); ax[1].legend()
fig.tight_layout()


## 7. Diagnostics versus energy

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.7))
ax[0].plot(results["energy_meV"]/hwc, results["nu"])
ax[0].set(xlabel=r"$E_F/(\hbar\omega_c)$", ylabel=r"$\nu$")
ax[0].grid(alpha=.25)

ax[1].plot(results["energy_meV"]/hwc, results["Rxy"], label=r"$R_{xy}$ barriers")
ax[1].plot(results["energy_meV"]/hwc, results["Rxx"], label=r"$R_{xx}$ barriers")
ax[1].plot(results0["energy_meV"]/hwc, results0["Rxy"], ls="--", label=r"$R_{xy}$ no barriers")
ax[1].plot(results0["energy_meV"]/hwc, results0["Rxx"], ls="--", label=r"$R_{xx}$ no barriers")
ax[1].set(xlabel=r"$E_F/(\hbar\omega_c)$", ylabel=r"$R\ [h/e^2]$")
ax[1].grid(alpha=.25); ax[1].legend(fontsize=8)
fig.tight_layout()


### Notes

- The two transport systems use the **same disorder array**, geometry, $B$, temperature, energy grid, filling calibration, and ideal leads. The only difference is the contact-barrier term.
- The barrier height is exactly `height_cyclotron × ħωc`; with the current parameters it is $\hbar\omega_c$.
- The common $\nu(E)$ calibration is obtained from the barrier-free bulk DOS and is used for both transport sweeps.
- Finite-temperature transport is handled identically in both cases by `qh6t.transport_sweep(..., T_K=T_K)`.
- `results` denotes the barrier case and `results0` the barrierless reference throughout the notebook.